## Here we plot the stacked images of the calibration source at 3 different frequencies

In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2
# Allows the figures to be dynamic
%matplotlib ipympl

In [ ]:
import numpy as np
import fitsio
import matplotlib.pyplot as plt
import matplotlib.mlab as mlab
from matplotlib import rc
plt.rc('figure',figsize=(6,6))
plt.rc('font',size=16)
rc('text',usetex=False)

In [ ]:
i_freqs_RGB = [0, 2, 4]
i_freqs = [0, 1, 2, 3, 4]
freq_map_list = [130, 140, 150, 160, 170]
method = "rms"
TESNum = 93 #93
n_freqs = len(i_freqs)

# Read maps created in All_scans_demodulation_src.ipynb
maps = []
azimuth = fitsio.read("test_maps/corr_azimuth_4{}.fits".format(method)) # azimuth if calsource is at zenith and then rotated at calsource position
elevation = fitsio.read("test_maps/corr_elevation_4{}.fits".format(method)) # elevation if calsource is at zenith and then rotated at calsource position
for i_freq in i_freqs:
    add_on = "_" + method + "_" + str(freq_map_list[i_freq]) + "GHz"
    maps.append(-fitsio.read("test_maps/corr_TES_{}_4{}.fits".format(TESNum, add_on)))
maps = np.array(maps)

In [ ]:
#https://stackoverflow.com/questions/10443295/combine-3-separate-numpy-arrays-to-an-rgb-image-in-python

maps_stacked_RGB = np.zeros((len(maps[0]), len(maps[0][0]), 3))
for i_freq in range(3):
    maps_stacked_RGB[..., i_freq] = maps[i_freqs_RGB[i_freq]]/np.max(maps[i_freq]) * 2 # the maps are already positive
# stacked_map = # RGB

In [ ]:
maps_stacked = np.zeros((len(maps[0]), len(maps[0][0]), n_freqs))
for i_freq in range(len(maps)):
    maps_stacked[..., i_freq] = maps[i_freq]/np.max(maps[i_freq]) * 2 # the maps are already positive
# stacked_map = # RGB

In [ ]:
print(np.shape(azimuth))

In [ ]:
size_map = 30 # deg (see ang_res definition in All_scans_demodulation_src.ipynb)
Npix = len(azimuth)
pixel_angle = np.linspace(0, size_map, Npix)
fig, ax = plt.subplots()
ax.imshow(maps_stacked_RGB)
ticks = np.arange(50, Npix, Npix//5)
xlabels = (pixel_angle[ticks] - pixel_angle[Npix//2]).astype(int)
# yticks = np.arange(50, Npix, Npix//5)
ylabels = (pixel_angle[ticks] - pixel_angle[Npix//2]).astype(int)
ax.set_xticks(ticks, labels=xlabels)
ax.set_yticks(ticks, labels=ylabels)
ax.set_xlabel("Distance to center [deg]")
ax.set_ylabel("Distance to center [deg]")
plt.tight_layout()
plt.savefig("figures/calsource_image_3freqs.pdf")
plt.show()

In [ ]:
# for each freq we save an image
size_map = 30 # deg (see ang_res definition in All_scans_demodulation_src.ipynb)
Npix = len(azimuth)
pixel_angle = np.linspace(0, size_map, Npix)
mult = [4, 1.5, 2, 1.5, 2]
for i_freq in range(n_freqs):
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(maps[i_freq]/np.max(maps[i_freq]) * mult[i_freq], vmin=0, vmax=1)
    ticks = np.arange(50, Npix, Npix//5)
    xlabels = (pixel_angle[ticks] - pixel_angle[Npix//2]).astype(int)
    # yticks = np.arange(50, Npix, Npix//5)
    ylabels = (pixel_angle[ticks] - pixel_angle[Npix//2]).astype(int)
    ax.set_xticks(ticks, labels=xlabels)
    ax.set_yticks(ticks, labels=ylabels)
    ax.set_xlabel("Distance to center [deg]")
    ax.set_ylabel("Distance to center [deg]")
    ax.set_title("Lab data, {}GHz".format(freq_map_list[i_freq]))
    plt.tight_layout()
    plt.savefig("figures/calsource_image_{}GHz.png".format(freq_map_list[i_freq]))
    plt.show()
    plt.close()

In [ ]:
# let's do a gif
# https://stackoverflow.com/questions/753190/programmatically-generate-video-or-animated-gif-in-python
import imageio

filenames = ["figures/calsource_image_{}GHz.png".format(freq_map_list[i_freq]) for i_freq in range(n_freqs)]

images = []
for filename in filenames:
    images.append(imageio.imread(filename))
imageio.mimsave("figures/animation_5freqs.gif", images, duration=800, loop=0)